# 92. The Location-Routing Problem (LRP)
## Tier 5: The Integrated Digital Twin

### Key assumptions
- Digital twin provides real-time synchronization with physical supply chain
- IoT sensors continuously monitor depot operations and vehicle movements
- Simulation engine enables what-if scenario analysis
- Predictive analytics forecast demand and disruption impacts
- Dynamic re-optimization responds to changing conditions

### Approach (step-by-step)
1. **Digital Twin Architecture**: Multi-layer system with physical-virtual synchronization
2. **IoT Sensor Integration**: Real-time data collection from depots and vehicles
3. **Simulation Engine**: High-fidelity modeling of supply chain dynamics
4. **Predictive Analytics**: Demand forecasting and disruption prediction
5. **Scenario Analysis**: What-if testing for strategic decisions
6. **Dynamic Optimization**: Continuous re-optimization based on real-time data

### What to look for in the results
- Real-time monitoring dashboard with live KPI tracking
- Scenario comparison showing impact of different decisions
- Predictive analytics accuracy and forecasting performance
- System resilience under disruption scenarios
- Cost-benefit analysis of digital twin implementation

### Concrete example (from the source)
- **Problem**: Extended 2-depot, 3-customer system with digital twin integration
- **Digital Twin**: 4-layer architecture with 500+ IoT sensors
- **Scenarios**: Fuel price increase, demand growth, depot disruptions
- **Expected Benefits**: 23% pick time reduction, 31% space utilization improvement

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
import math
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

In [ ]:
@dataclass
class LRPInstance:
    """Data class for Location-Routing Problem instance"""
    customers: List[int]
    depots: List[int]
    vehicles: List[int]
    depot_costs: Dict[int, float]
    demands: Dict[int, float]
    vehicle_capacity: float
    travel_costs: Dict[Tuple[int, int], float]
    
    def get_all_nodes(self):
        return self.customers + self.depots

@dataclass
class IoTSensor:
    """IoT sensor for monitoring supply chain assets"""
    sensor_id: str
    sensor_type: str  # 'depot_capacity', 'vehicle_location', 'demand_forecast', 'traffic'
    location: int     # Node ID this sensor monitors
    reading_value: float
    timestamp: datetime
    accuracy: float   # Sensor accuracy (0-1)

@dataclass
class DigitalTwinState:
    """Current state of the digital twin system"""
    current_time: datetime
    depot_utilization: Dict[int, float]
    vehicle_locations: Dict[int, Tuple[int, int]]  # vehicle_id -> (current_node, next_node)
    demand_forecasts: Dict[int, float]
    traffic_conditions: Dict[Tuple[int, int], float]
    system_kpis: Dict[str, float]

# Create the concrete example
def create_concrete_example():
    customers = [1, 2, 3]
    depots = [4, 5]
    vehicles = [1, 2]
    
    depot_costs = {4: 100, 5: 120}
    demands = {1: 10, 2: 15, 3: 20}
    vehicle_capacity = 30
    
    travel_costs = {
        (1, 2): 15, (2, 1): 15,
        (1, 3): 20, (3, 1): 20,
        (2, 3): 18, (3, 2): 18,
        (4, 1): 12, (1, 4): 12,
        (4, 2): 14, (2, 4): 14,
        (4, 3): 25, (3, 4): 25,
        (5, 1): 18, (1, 5): 18,
        (5, 2): 16, (2, 5): 16,
        (5, 3): 22, (3, 5): 22,
        (4, 5): 30, (5, 4): 30,
        (1, 1): 0, (2, 2): 0, (3, 3): 0,
        (4, 4): 0, (5, 5): 0
    }
    
    return LRPInstance(
        customers=customers,
        depots=depots,
        vehicles=vehicles,
        depot_costs=depot_costs,
        demands=demands,
        vehicle_capacity=vehicle_capacity,
        travel_costs=travel_costs
    )

instance = create_concrete_example()
print(f"LRP Instance created for Digital Twin integration:")
print(f"Customers: {instance.customers}")
print(f"Potential Depots: {instance.depots}")
print(f"Vehicles: {instance.vehicles}")
print(f"Vehicle Capacity: {instance.vehicle_capacity}")

LRP Instance created for Digital Twin integration:
Customers: [1, 2, 3]
Potential Depots: [4, 5]
Vehicles: [1, 2]
Vehicle Capacity: 30


In [2]:
class IoTSensorNetwork:
    """IoT sensor network for real-time monitoring"""
    
    def __init__(self, instance):
        self.instance = instance
        self.sensors = []
        self.initialize_sensors()
    
    def initialize_sensors(self):
        """Initialize IoT sensors for all nodes and vehicles"""
        
        # Depot sensors (capacity utilization)
        for depot in self.instance.depots:
            for i in range(3):  # 3 sensors per depot
                sensor = IoTSensor(
                    sensor_id=f"DEPOT_{depot}_{i}",
                    sensor_type="depot_capacity",
                    location=depot,
                    reading_value=0.0,
                    timestamp=datetime.now(),
                    accuracy=0.95
                )
                self.sensors.append(sensor)
        
        # Customer demand sensors
        for customer in self.instance.customers:
            sensor = IoTSensor(
                sensor_id=f"DEMAND_{customer}",
                sensor_type="demand_forecast",
                location=customer,
                reading_value=self.instance.demands[customer],
                timestamp=datetime.now(),
                accuracy=0.90
            )
            self.sensors.append(sensor)
        
        # Vehicle location sensors
        for vehicle in self.instance.vehicles:
            sensor = IoTSensor(
                sensor_id=f"VEHICLE_{vehicle}",
                sensor_type="vehicle_location",
                location=vehicle,
                reading_value=0.0,  # Will be updated with actual location
                timestamp=datetime.now(),
                accuracy=0.98
            )
            self.sensors.append(sensor)
        
        # Traffic condition sensors
        for u in self.instance.get_all_nodes():
            for v in self.instance.get_all_nodes():
                if u != v and (u, v) in self.instance.travel_costs:
                    sensor = IoTSensor(
                        sensor_id=f"TRAFFIC_{u}_{v}",
                        sensor_type="traffic",
                        location=u,  # Monitors traffic from u to v
                        reading_value=1.0,  # 1.0 = normal traffic
                        timestamp=datetime.now(),
                        accuracy=0.85
                    )
                    self.sensors.append(sensor)
    
    def update_sensor_readings(self, current_state, simulation_time):
        """Update all sensor readings based on current state"""
        
        for sensor in self.sensors:
            sensor.timestamp = simulation_time
            
            if sensor.sensor_type == "depot_capacity":
                # Simulate depot utilization
                base_utilization = current_state.depot_utilization.get(sensor.location, 0.5)
                noise = np.random.normal(0, 0.05)  # Sensor noise
                sensor.reading_value = np.clip(base_utilization + noise, 0, 1)
            
            elif sensor.sensor_type == "demand_forecast":
                # Simulate demand forecasting with some variation
                base_demand = current_state.demand_forecasts.get(sensor.location, 10)
                forecast_variation = np.random.normal(0, base_demand * 0.1)
                sensor.reading_value = max(5, base_demand + forecast_variation)
            
            elif sensor.sensor_type == "vehicle_location":
                # Update vehicle location
                vehicle_loc = current_state.vehicle_locations.get(sensor.location, (4, 1))
                sensor.reading_value = vehicle_loc[0]  # Current node
            
            elif sensor.sensor_type == "traffic":
                # Update traffic conditions
                route_key = tuple(map(int, sensor.sensor_id.split('_')[1:]))
                base_traffic = current_state.traffic_conditions.get(route_key, 1.0)
                traffic_noise = np.random.normal(0, 0.1)
                sensor.reading_value = max(0.5, base_traffic + traffic_noise)
    
    def get_sensor_data_by_type(self, sensor_type):
        """Get all sensors of a specific type"""
        return [s for s in self.sensors if s.sensor_type == sensor_type]
    
    def get_sensor_readings_summary(self):
        """Get summary of current sensor readings"""
        summary = {
            'total_sensors': len(self.sensors),
            'depot_sensors': len(self.get_sensor_data_by_type('depot_capacity')),
            'demand_sensors': len(self.get_sensor_data_by_type('demand_forecast')),
            'vehicle_sensors': len(self.get_sensor_data_by_type('vehicle_location')),
            'traffic_sensors': len(self.get_sensor_data_by_type('traffic')),
            'average_accuracy': np.mean([s.accuracy for s in self.sensors])
        }
        return summary

print("IoT Sensor Network class implemented")

IoT Sensor Network class implemented


In [3]:
class SimulationEngine:
    """High-fidelity simulation engine for digital twin"""
    
    def __init__(self, instance):
        self.instance = instance
        self.simulation_time = datetime.now()
        self.time_step = timedelta(minutes=15)  # 15-minute time steps
        self.current_state = self.initialize_state()
    
    def initialize_state(self):
        """Initialize the digital twin state"""
        
        return DigitalTwinState(
            current_time=self.simulation_time,
            depot_utilization={depot: 0.5 for depot in self.instance.depots},
            vehicle_locations={vehicle: (4, 1) for vehicle in self.instance.vehicles},  # All at depot 4
            demand_forecasts=self.instance.demands.copy(),
            traffic_conditions={(u, v): 1.0 for u, v in self.instance.travel_costs if u != v},
            system_kpis={
                'total_cost': 0.0,
                'service_level': 0.95,
                'vehicle_utilization': 0.7,
                'depot_utilization': 0.5,
                'on_time_delivery': 0.92
            }
        )

print("Simulation Engine class implemented")

Simulation Engine class implemented


In [4]:
class PredictiveAnalytics:
    """Predictive analytics module for demand forecasting and disruption prediction"""
    
    def __init__(self, instance):
        self.instance = instance
        self.historical_data = self.generate_historical_data()
    
    def generate_historical_data(self, days=30):
        """Generate synthetic historical data for training"""
        
        historical_data = []
        
        for day in range(days):
            daily_data = {
                'date': datetime.now() - timedelta(days=days - day),
                'demands': {},
                'disruptions': [],
                'weather_conditions': np.random.choice(['sunny', 'rainy', 'cloudy'], p=[0.6, 0.2, 0.2]),
                'fuel_price': np.random.normal(1.0, 0.1)  # Normalized fuel price
            }
            
            # Generate demand data with patterns
            for customer in self.instance.customers:
                base_demand = self.instance.demands[customer]
                # Add weekly pattern (higher on weekdays)
                day_of_week = daily_data['date'].weekday()
                if day_of_week < 5:  # Weekday
                    weekly_multiplier = 1.2
                else:  # Weekend
                    weekly_multiplier = 0.8
                
                # Add seasonal trend
                seasonal_multiplier = 1.0 + 0.1 * np.sin(day * np.pi / 15)
                
                # Random variation
                random_variation = np.random.normal(0, base_demand * 0.15)
                
                final_demand = base_demand * weekly_multiplier * seasonal_multiplier + random_variation
                daily_data['demands'][customer] = max(5, final_demand)
            
            # Occasionally add disruptions
            if np.random.random() < 0.1:  # 10% chance of disruption
                disruption_types = ['depot_maintenance', 'vehicle_breakdown', 'road_closure', 'supplier_delay']
                disruption = {
                    'type': np.random.choice(disruption_types),
                    'duration': np.random.randint(1, 8),  # hours
                    'impact': np.random.uniform(0.1, 0.8)
                }
                daily_data['disruptions'].append(disruption)
            
            historical_data.append(daily_data)
        
        return historical_data

print("Predictive Analytics class implemented")

Predictive Analytics class implemented


In [5]:
class DigitalTwinSystem:
    """Integrated Digital Twin System for LRP"""
    
    def __init__(self, instance):
        self.instance = instance
        self.sensor_network = IoTSensorNetwork(instance)
        self.simulation_engine = SimulationEngine(instance)
        self.predictive_analytics = PredictiveAnalytics(instance)
        
        # System state
        current_time = datetime.now()
        self.current_state = DigitalTwinState(
            current_time=current_time,
            depot_utilization={depot: 0.5 for depot in instance.depots},
            vehicle_locations={vehicle: (4, 1) for vehicle in instance.vehicles},
            demand_forecasts=instance.demands.copy(),
            traffic_conditions={(u, v): 1.0 for u, v in instance.travel_costs if u != v},
            system_kpis={
                'total_cost': 0.0,
                'service_level': 0.95,
                'vehicle_utilization': 0.7,
                'depot_utilization': 0.5,
                'on_time_delivery': 0.92
            }
        )

print("Digital Twin System class implemented")

Digital Twin System class implemented


In [ ]:
# Initialize the Digital Twin System
digital_twin = DigitalTwinSystem(instance)

print("Digital Twin System initialized")
print(f"IoT Sensors: {digital_twin.sensor_network.get_sensor_readings_summary()['total_sensors']}")
print(f"Historical Data Points: {len(digital_twin.predictive_analytics.historical_data)}")

Digital Twin System initialized
IoT Sensors: 31
Historical Data Points: 30


In [ ]:
def visualize_digital_twin_architecture():
    """Visualize the digital twin architecture"""
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # Create architecture diagram
    layers = [
        {'name': 'Physical World', 'y': 0.9, 'components': ['Depots', 'Vehicles', 'Customers', 'Routes']},
        {'name': 'IoT Sensor Layer', 'y': 0.7, 'components': ['500+ Sensors', 'Real-time Data', 'Edge Computing']},
        {'name': 'Digital Twin Core', 'y': 0.5, 'components': ['State Management', 'Simulation Engine', 'Analytics']},
        {'name': 'Application Layer', 'y': 0.3, 'components': ['Monitoring', 'Optimization', 'Decision Support']},
        {'name': 'User Interface', 'y': 0.1, 'components': ['Dashboard', 'Alerts', 'Reports']}
    ]
    
    # Draw layers
    for layer in layers:
        # Draw layer box
        rect = plt.Rectangle((0.1, layer['y'] - 0.08), 0.8, 0.06, 
                           facecolor='lightblue', edgecolor='navy', linewidth=2)
        ax.add_patch(rect)
        
        # Add layer name
        ax.text(0.5, layer['y'], layer['name'], ha='center', va='center', 
                fontsize=12, fontweight='bold')
        
        # Add components
        components_text = ', '.join(layer['components'])
        ax.text(0.5, layer['y'] - 0.04, components_text, ha='center', va='center', 
                fontsize=8, style='italic')
    
    # Draw connections
    for i in range(len(layers) - 1):
        y1 = layers[i]['y'] - 0.08
        y2 = layers[i + 1]['y'] + 0.02
        ax.arrow(0.5, y1, 0, y2 - y1, head_width=0.02, head_length=0.01, 
                fc='red', ec='red', alpha=0.7)
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title('Digital Twin Architecture', fontsize=16, fontweight='bold')
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize architecture
visualize_digital_twin_architecture()


=== PSO Performance Analysis ===
Solution Quality: $20,000 total cost
Average Turnaround: 6.7 hours per vessel
Crane Utilization: 15 cranes assigned
Convergence Speed: 11 iterations
Computational Efficiency: 2.14 seconds
Swarm Diversity: 490.53 (exploration capability)


In [ ]:
def demonstrate_real_time_monitoring():
    """Demonstrate real-time monitoring capabilities"""
    
    print("\n" + "="*60)
    print("REAL-TIME MONITORING DEMONSTRATION")
    print("="*60)
    
    # Simulate 6 hours of monitoring
    monitoring_data = []
    current_time = datetime.now()
    
    for hour in range(6):
        simulation_time = current_time + timedelta(hours=hour)
        
        # Update sensor readings
        digital_twin.sensor_network.update_sensor_readings(digital_twin.current_state, simulation_time)
        
        # Collect monitoring data
        hour_data = {
            'timestamp': simulation_time,
            'depot_utilization': digital_twin.current_state.depot_utilization.copy(),
            'system_kpis': digital_twin.current_state.system_kpis.copy(),
            'sensor_summary': digital_twin.sensor_network.get_sensor_readings_summary()
        }
        
        monitoring_data.append(hour_data)
        
        # Update state with realistic variations
        for depot in instance.depots:
            current_util = digital_twin.current_state.depot_utilization[depot]
            change = np.random.normal(0, 0.05)
            digital_twin.current_state.depot_utilization[depot] = np.clip(current_util + change, 0, 1)
        
        print(f"Hour {hour}: Depot J4 Utilization = {digital_twin.current_state.depot_utilization[4]:.1%}, "
              f"Sensor Health = {digital_twin.sensor_network.get_sensor_readings_summary()['average_accuracy']:.1%}")
    
    return monitoring_data

# Run monitoring demonstration
monitoring_results = demonstrate_real_time_monitoring()


REAL-TIME MONITORING DEMONSTRATION
📈 Progress: 13/61 (21.3%)
✅ Success: 13
🚀 Executing P58-Tier-1_executed.ipynb...
Hour 0: Depot J4 Utilization = 56.3%, Sensor Health = 88.3%


In [ ]:
def visualize_monitoring_results(monitoring_data):
    """Visualize monitoring results"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    hours = list(range(len(monitoring_data)))
    
    # Plot 1: Depot utilization
    for depot in instance.depots:
        utilization_series = [data['depot_utilization'][depot] for data in monitoring_data]
        ax1.plot(hours, utilization_series, label=f'Depot J{depot-3}', marker='o')
    
    ax1.set_xlabel('Hour')
    ax1.set_ylabel('Utilization (%)')
    ax1.set_title('Real-Time Depot Utilization')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: System KPIs
    kpi_names = ['service_level', 'vehicle_utilization', 'on_time_delivery']
    for kpi in kpi_names:
        kpi_series = [data['system_kpis'][kpi] for data in monitoring_data]
        ax2.plot(hours, kpi_series, label=kpi.replace('_', ' ').title())
    
    ax2.set_xlabel('Hour')
    ax2.set_ylabel('KPI Value')
    ax2.set_title('System Performance KPIs')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Sensor network health
    sensor_health = [data['sensor_summary']['average_accuracy'] for data in monitoring_data]
    ax3.plot(hours, sensor_health, 'g-', marker='s')
    ax3.set_xlabel('Hour')
    ax3.set_ylabel('Average Sensor Accuracy')
    ax3.set_title('IoT Sensor Network Health')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Total sensors by type
    final_summary = monitoring_data[-1]['sensor_summary']
    sensor_types = ['depot_sensors', 'demand_sensors', 'vehicle_sensors', 'traffic_sensors']
    sensor_counts = [final_summary[stype] for stype in sensor_types]
    labels = ['Depot', 'Demand', 'Vehicle', 'Traffic']
    
    ax4.pie(sensor_counts, labels=labels, autopct='%1.1f%%')
    ax4.set_title('Sensor Distribution')
    
    plt.tight_layout()
    plt.show()

# Visualize monitoring results
visualize_monitoring_results(monitoring_results)

Iteration  50: Best Fitness = 20000.00, Diversity = 426.32
Iteration 100: Best Fitness = 20000.00, Diversity = 278.06


### Why this Tier exists vs earlier Tiers
The Integrated Digital Twin represents a paradigm shift from offline optimization to real-time, data-driven decision making:

**Tier 1-4 Limitations:**
- ❌ Static optimization based on historical data
- ❌ No real-time monitoring or adaptation
- ❌ Limited scenario analysis capabilities
- ❌ No predictive analytics or foresight
- ❌ Reactive rather than proactive decision making

**Tier 5 Advantages:**
- ✅ **Real-time synchronization** with physical operations
- ✅ **IoT sensor network** providing continuous data streams
- ✅ **Predictive analytics** for demand forecasting and disruption prediction
- ✅ **Scenario simulation** for strategic decision support
- ✅ **Dynamic re-optimization** based on current conditions
- ✅ **System-wide visibility** across all operations

### Pros / Cons vs earlier Tiers

**Pros:**
- ✅ **Real-time insights** - immediate visibility into operations
- ✅ **Predictive capabilities** - forecast future conditions and disruptions
- ✅ **Scenario testing** - evaluate decisions before implementation
- ✅ **Continuous improvement** - ongoing optimization based on live data
- ✅ **System integration** - holistic view of entire supply chain
- ✅ **Proactive management** - anticipate and prevent issues

**Cons:**
- ❌ **Implementation complexity** - requires significant technical infrastructure
- ❌ **High investment costs** - IoT sensors, computing resources, expertise
- ❌ **Data management challenges** - handling large volumes of real-time data
- ❌ **Integration requirements** - connecting with existing systems
- ❌ **Maintenance overhead** - continuous system monitoring and updates
- ❌ **Security concerns** - protecting sensitive operational data

### When to use this Tier
- **Large-scale operations** where real-time visibility provides significant value
- **High-value supply chains** where optimization improvements justify investment
- **Complex networks** with multiple interdependencies and constraints
- **Dynamic environments** with frequent changes and disruptions
- **Strategic planning** requiring comprehensive scenario analysis
- **Digital transformation initiatives** aiming for operational excellence

### Key Digital Twin Innovations

1. **Multi-Layer Architecture**: Physical → Sensors → Digital Core → Applications → Users

2. **IoT Sensor Network**: 500+ sensors providing real-time operational data

3. **Predictive Analytics**: Demand forecasting and disruption risk prediction

4. **Scenario Simulation**: What-if analysis for strategic decision support

5. **Real-Time Optimization**: Continuous re-optimization based on current conditions

6. **System Integration**: Holistic view across depots, vehicles, customers, and routes

The Integrated Digital Twin transforms supply chain management from reactive optimization to proactive, data-driven decision making, enabling organizations to anticipate changes, optimize continuously, and achieve operational excellence through real-time visibility and predictive insights.